# 0.5 — Auto-label Open Images with a pretrained YOLO

## Purpose

This notebook runs a pretrained COCO YOLO over the Open Images images in our
final dataset to **add missing bounding boxes** (pseudo-labeling /
densification) to the existing YOLO `.txt` labels. It post-processes the
DVC-packaged dataset (`datasets/table_assistant_yolo_package.zip`) so that
under-annotated Open Images frames gain the tabletop objects a human labeler
missed.

What it does / does NOT do:

- Densifies labels **only** for Open Images-sourced images.
- Does **NOT** touch UEC food images or their labels.
- Does **NOT** retrain anything; it only uses a pretrained model for inference.
- Adds new boxes to existing labels; it does not remove or relabel existing ones.

Known limitation: COCO has no flat `plate` class (only `bowl`), so `plate`
densification is inherently bounded by what the COCO model can detect.

## 1. Repository setup

In [1]:
from pathlib import Path

REPO_URL = "https://github.com/LucasGVallejos/iaa-visual-table-assistant.git"
REPO_DIR = Path("/content/iaa-visual-table-assistant")

%cd /content

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repository already present, pulling latest changes...")
    !git -C {REPO_DIR} pull --ff-only

%cd {REPO_DIR}

[Errno 2] No such file or directory: '/content'
/Users/alexanderarmua/Projects/UTN/iaa-visual-table-assistant/notebooks
fatal: could not create leading directories of '/content/iaa-visual-table-assistant': Read-only file system
[Errno 2] No such file or directory: '/content/iaa-visual-table-assistant'
/Users/alexanderarmua/Projects/UTN/iaa-visual-table-assistant/notebooks


## 2. Dependencies and GPU check

In [2]:
!pip install -q -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [3]:
!nvidia-smi

zsh:1: command not found: nvidia-smi


In [4]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print(
        "[WARN] CUDA is not available in this runtime. YOLO will run on CPU "
        "and inference will be extremely slow. "
        "Switch to a GPU runtime: Runtime > Change runtime type > GPU."
    )

CUDA available: False
[WARN] CUDA is not available in this runtime. YOLO will run on CPU and inference will be extremely slow. Switch to a GPU runtime: Runtime > Change runtime type > GPU.


## 3. Google Drive mount and DVC credentials

In [5]:
from google.colab import drive
drive.mount("/content/drive")

ModuleNotFoundError: No module named 'google.colab'

In [6]:
from google.colab import userdata
import os

gdrive_credentials = userdata.get("GDRIVE_CREDENTIALS_DATA")

if not gdrive_credentials:
    raise RuntimeError(
        "Missing Colab Secret: GDRIVE_CREDENTIALS_DATA. "
        "Create it from the cached DVC Google Drive credentials JSON."
    )

os.environ["GDRIVE_CREDENTIALS_DATA"] = gdrive_credentials

print("DVC Google Drive credentials loaded from Colab Secret.")

ModuleNotFoundError: No module named 'google.colab'

## 4. Pull and restore the packaged dataset

Pulls the DVC-tracked dataset package from the gdrive remote and restores it (symlinking `datasets/table_assistant_yolo` to the package's inner dataset dir); requires the credentials cell above to have run.

In [ ]:
!dvc pull datasets/table_assistant_yolo_package.zip.dvc

In [ ]:
!python -m src.data.preparation.restore_dataset_package

## 5. Inspect the manifest and a few Open Images samples

Confirms the package manifest reads and the Open Images images render with their current boxes (a sanity check before any labeling).

In [ ]:
!python -m src.data.auto_label.manifest

In [ ]:
!python -m src.data.auto_label.inspect_open_images --samples 3

from pathlib import Path
from IPython.display import Image, display

from src.utils.paths import get_outputs_dir

checks_dir = get_outputs_dir() / "auto_label_checks" / "phase3_existing"
for png in sorted(checks_dir.glob("*.png")):
    display(Image(str(png)))